Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC conn = trino.dbapi.connect(
# MAGIC     host="trino-tableau-prod.dp.navi-tech.in",
# MAGIC     port=443,
# MAGIC     user="databricks-user",
# MAGIC     catalog="awsdatacatalog",
# MAGIC     schema="default",
# MAGIC     http_scheme="https",
# MAGIC )
# MAGIC
# MAGIC query = """
# MAGIC WITH prompts AS (
# MAGIC     SELECT ticket_id, prompt, created_at FROM corp_analytics.krisha_chat_ct_tbl
# MAGIC     UNION
# MAGIC     SELECT ticket_id, prompt, created_at FROM corp_analytics.trisha_ct_chat_tbl
# MAGIC     UNION
# MAGIC     SELECT ticket_id, prompt, created_at FROM corp_analytics.esha_ct_tbl
# MAGIC ),   
# MAGIC prompts1 AS (
# MAGIC     SELECT
# MAGIC         ticket_id,
# MAGIC         MAX_BY(SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1), created_at) AS last_prompt,
# MAGIC         MIN_BY(SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1), created_at) AS first_prompt
# MAGIC     FROM prompts
# MAGIC     WHERE SPLIT_PART(SPLIT_PART(prompt, 'Customer query categories List: [', 2), ']', 1) IS NOT NULL
# MAGIC     GROUP BY 1
# MAGIC ),
# MAGIC
# MAGIC base_data AS (
# MAGIC SELECT
# MAGIC     a.canonical_ticket_id,
# MAGIC     a.phone_number,
# MAGIC     a.name,
# MAGIC     a.email,
# MAGIC     a.user_selected_language,
# MAGIC     a.csat,
# MAGIC     CAST(a.first_agent_assigned_time AS timestamp) AS first_agent_assigned_time,
# MAGIC     CAST(a.resolved_time AS timestamp) AS resolved_time,
# MAGIC     a.overall_last_agent_type_for_sub_ticket AS overall_last_agent_type,
# MAGIC     a.ticket_group,
# MAGIC     CASE 
# MAGIC         WHEN overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' AND obcs_l1_agent_disposition IS NOT NULL THEN obcs_l1_agent_disposition
# MAGIC         WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' AND l1_agent_disposition IS NOT NULL THEN l1_agent_disposition
# MAGIC         WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' AND l2_agent_disposition IS NOT NULL THEN l2_agent_disposition
# MAGIC         ELSE xz.origination_reason
# MAGIC     END AS agent_filled_reason,
# MAGIC     CASE 
# MAGIC         WHEN overall_last_agent_type_for_sub_ticket LIKE '%OUTBOUND%' AND obcs_l1_agent_sub_disposition IS NOT NULL THEN obcs_l1_agent_sub_disposition
# MAGIC         WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L1' AND l1_agent_sub_disposition IS NOT NULL THEN l1_agent_sub_disposition
# MAGIC         WHEN overall_last_agent_type_for_sub_ticket = 'SUPPORT_AGENT_L2' AND l2_agent_sub_disposition IS NOT NULL THEN l2_agent_sub_disposition
# MAGIC         ELSE xz.origination_sub_reason
# MAGIC     END AS agent_filled_sub_reason,
# MAGIC     a.last_l1_agent_name AS l1_agent_name,
# MAGIC     a.last_l2_agent_name AS l2_agent_name,
# MAGIC     a.bot_description,
# MAGIC     xz.origination_reason AS agent_disposition,
# MAGIC     xz.origination_sub_reason AS agent_sub_disposition,
# MAGIC     p1.first_prompt AS language_sel,
# MAGIC     a.ticket_created_time,
# MAGIC     a.ticket_state,
# MAGIC     JSON_EXTRACT_SCALAR(xz.meta_data, '$.source_screen') AS required_screen,
# MAGIC
# MAGIC     CAST(
# MAGIC         (date_diff('second', CAST(a.first_agent_assigned_time AS timestamp), 
# MAGIC          CAST(a.resolved_time AS timestamp)) / 60) AS INTEGER
# MAGIC     ) AS aht_minutes,
# MAGIC     
# MAGIC     CAST(
# MAGIC         (date_diff('second', CAST(a.first_agent_assigned_time AS timestamp), 
# MAGIC          CAST(a.resolved_time AS timestamp)) % 60) AS INTEGER
# MAGIC     ) AS aht_seconds,
# MAGIC
# MAGIC     ROW_NUMBER() OVER (
# MAGIC         PARTITION BY a.canonical_ticket_id 
# MAGIC         ORDER BY a.ticket_created_time DESC
# MAGIC     ) AS rn
# MAGIC
# MAGIC FROM corp_analytics.cx_sub_ticket_data_model_tbl a
# MAGIC LEFT JOIN crm_tickets.tickets xz ON a.ticket_id = xz.id
# MAGIC LEFT JOIN prompts1 p1 ON a.ticket_id = p1.ticket_id
# MAGIC
# MAGIC WHERE 
# MAGIC     -- MTD: ticket_created_time in current month
# MAGIC     DATE(SUBSTRING(a.ticket_created_time, 1, 10)) >= CAST(DATE_TRUNC('month', current_date) AS date)
# MAGIC     AND DATE(SUBSTRING(a.ticket_created_time, 1, 10)) <= current_date
# MAGIC
# MAGIC     -- Agent Type Filter
# MAGIC     AND a.overall_last_agent_type_for_sub_ticket IN 
# MAGIC         ('SUPPORT_AGENT_L1','SUPPORT_AGENT_L2')
# MAGIC
# MAGIC     -- Ticket Group Filter
# MAGIC     AND a.ticket_group IN (
# MAGIC         'BBPS',
# MAGIC         'PERSONAL_LOAN',
# MAGIC         'DIGITAL_GOLD',
# MAGIC         'UPI',
# MAGIC         'CREDIT_LINE_UPI',
# MAGIC         'AMC',
# MAGIC         'GENERAL_INSURANCE',
# MAGIC         'HOME_LOAN_SALES'
# MAGIC     )
# MAGIC
# MAGIC     AND a.source_channel = 'CHAT'
# MAGIC )
# MAGIC
# MAGIC SELECT 
# MAGIC     agent_name,
# MAGIC     ticket_group,
# MAGIC     COUNT(*) AS tickets_handled_mtd
# MAGIC FROM (
# MAGIC     SELECT 
# MAGIC         COALESCE(l1_agent_name, l2_agent_name, 'Unknown') AS agent_name,
# MAGIC         ticket_group
# MAGIC     FROM base_data
# MAGIC     WHERE rn = 1
# MAGIC ) t
# MAGIC GROUP BY agent_name, ticket_group
# MAGIC ORDER BY agent_name, tickets_handled_mtd DESC
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df = pd.read_sql(query, conn)
# MAGIC
# MAGIC if df.empty:
# MAGIC     print("Query returned no results. The DataFrame is empty.")
# MAGIC else:
# MAGIC     # Convert call_Time from datetime.time to string to avoid Spark conversion error
# MAGIC     df['call_Time'] = df['call_Time'].astype(str)
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1sshtYrCvZS7lLufFwacEu8ZczfHmk4_U9QpUeFnPTb4/edit?gid=1963761199#gid=1963761199"
worksheet_name = "Agent Wise MTD Tickets Handled"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)